In [54]:
import polars as pl
import pandas as pd
import numpy as mp
import altair as alt
import pyarrow
from polars import Null

from func.UsefulFunctions import *

This notebook documents the cleaning and inspection process for the artist data on wikidata

In [55]:
artistDF = pl.read_parquet("./Data/pipelinesArtistsWikidata.parquet")
print(f"There are {len(artistDF["artist"].unique())} unique artist entities after the removal of the disallowed occupations.")
display(artistDF.head(3))

There are 444888 unique artist entities after the removal of the disallowed occupations.


artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str
"""Q7161""","""Q36834""","""Q6581097""","""composer""","""male""","""Q29""",null,null,null,"""Q1413570""","""https://en.wikipedia.org/wiki/…","""533761c0-61cd-4d0e-b264-c71d5d…",null,null,"""Spain""","""sardana""","""José María Ventura Casas"""
"""Q7241""","""Q36834""","""Q6581097""","""composer""","""male""","""Q129286""",null,null,538.0,null,"""https://en.wikipedia.org/wiki/…","""13274b60-181b-431a-b049-b48560…",null,null,"""British Raj""",null,"""Rabindranath Tagore"""
"""Q7241""","""Q36834""","""Q6581097""","""composer""","""male""","""Q129286""",null,null,538.0,null,"""https://en.wikipedia.org/wiki/…","""13274b60-181b-431a-b049-b48560…",null,null,"""British Raj""",null,"""Rabindranath Tagore"""


In [56]:
#inspect the different fields in the system that are error prone
#see the different genders in the dataframe
print(artistDF[["artist","genderLabel"]].unique().select(pl.col("genderLabel").value_counts(sort=True)).unnest("genderLabel"))
print(artistDF[["genderLabel"]].unique().select(pl.col("genderLabel")).to_series().to_list())

#see some genders who have unknown or none gender
noGender = artistDF.filter(
    pl.col("gender").is_null()
)
noGender = noGender.select(pl.col(["artist","gender","genderLabel"])).unique()
print(noGender)
#Save the file for individuals who do not have a gender
noGender.write_csv("./missingValuesStore/artistsEntitiesWithoutGenders.csv",)


shape: (30, 2)
┌─────────────────┬────────┐
│ genderLabel     ┆ count  │
│ ---             ┆ ---    │
│ str             ┆ u32    │
╞═════════════════╪════════╡
│ male            ┆ 331681 │
│ female          ┆ 112112 │
│ non-binary      ┆ 472    │
│ trans woman     ┆ 471    │
│ trans man       ┆ 107    │
│ …               ┆ …      │
│ gender agnostic ┆ 1      │
│ demigirl        ┆ 1      │
│ neutral sex     ┆ 1      │
│ apagender       ┆ 1      │
│ demiboy         ┆ 1      │
└─────────────────┴────────┘
['intersex', 'bigender', 'cisgender woman', 'male', 'non-binary', 'genderqueer', 'demigirl', 'travesti', 'intersex trans woman', 'demiboy', 'transfeminine', 'takatāpui', 'apagender', 'non-binary woman', 'transmasculine', 'agender', 'trans man', 'intersex woman', 'gender agnostic', 'transgender', 'trans woman', 'cisgender man', 'genderfluid', 'female', 'undisclosed gender', 'neutral sex', 'androgyne', 'intersex man', 'two-spirit', None]
shape: (41, 3)
┌────────────┬────────┬─────────────┐

Checked the genders to ensure that error/unknown genders were successfully nulled.

In [57]:
#Null the artist labels for the individuals who do not have an artist label

noNames = artistDF.filter(
    pl.col("artist").is_in(pl.col("artistLabel").implode())
).select(pl.col("artistLabel")).unique()
name_dict = {k: None for k in noNames["artistLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("artistLabel").replace(name_dict))

noNamesDF = artistDF.filter(
    pl.col("artistLabel").is_null(),
).select(pl.col(["artist","artistLabel"])).unique()
display(noNamesDF)
print(f"There are {len(noNamesDF)} artists with no default labels/labels in english.")
noNamesDF.write_csv("./missingValuesStore/artistsEntitiesWithoutLabels.csv")

artist,artistLabel
str,str
"""Q18238859""",null
"""Q22918495""",null
"""Q136747181""",null
"""Q4330878""",null
"""Q50640654""",null
…,…
"""Q139466986""",null
"""Q31825920""",null
"""Q56352168""",null


There are 14131 artists with no default labels/labels in english.


In [58]:
#The individuals with a country without an english label
#First let's see if there are any unknown or erroneous countries by using the Q-numbers and there are
artistDF.filter(
    pl.col('countryCitizenshipLabel').str.starts_with("Q")
).select(pl.col('countryCitizenshipLabel')).unique().to_series().to_list()

noLabel_Country = artistDF.filter(
    pl.col("countryCitizenship").is_in(pl.col("countryCitizenshipLabel").implode())
).select(pl.col("countryCitizenshipLabel")).unique()
countryName_dict = {k: None for k in noLabel_Country["countryCitizenshipLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("countryCitizenshipLabel").replace(countryName_dict))
artistDF = artistDF.with_columns(
    countryCitizenship = pl.when(~pl.col('countryCitizenship').str.starts_with("Q")).then(None).otherwise(pl.col('countryCitizenship'))
)
noCountryLabel = artistDF.filter(
    pl.col("countryCitizenshipLabel").is_null(),
).select(pl.col(["countryCitizenship","countryCitizenshipLabel"])).unique()
display(noCountryLabel)
# noCountry = artistDF.filter(
#     pl.col("countryCitizenship").is_in(pl.col("countryCitizenshipLabel").implode())
# ).select(pl.col("countryCitizenship")).unique()
# display(noCountry)
# print(f"There are {len(noCountry)} artists with a country with no labels. These will be cleaned out.")
# noCountry.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledCountries.csv")

artist,artistLabel,countryCitizenship,countryCitizenshipLabel
str,str,str,str
"""Q52771231""","""Donia Surowiec""",null,null
"""Q126688770""","""Sophia Lefeber""",null,null
"""Q139434373""",null,null,null
"""Q27553510""","""Giovanni Battista Mariani""",null,null
"""Q63191295""","""Asako Hirabayashi""",null,null
…,…,…,…
"""Q111454125""","""Joan Bluteau""",null,null
"""Q16704704""","""Yosyp Levytskyi""",null,null
"""Q114237189""","""Marjana Semkina""",null,null


In [59]:
#Genres with no labels in english/default language
noGenre = artistDF.filter(
    pl.col("genre").is_in(pl.col("genreLabel").implode())
).select(pl.col('genreLabel')).unique()
noGenreName_dict = {k: None for k in noGenre["genreLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("genreLabel").replace(noGenreName_dict))
artistDF = artistDF.with_columns(
    genre = pl.when(~pl.col('genre').str.starts_with("Q")).then(None).otherwise(pl.col('genre'))
)

noGenreLabel = artistDF.filter(
    pl.col("genreLabel").is_null(),
).select(pl.col(["artist","artistLabel","genre","genreLabel"])).unique()
display(noGenreLabel)
#print(f"There are {len(noGenre)} artists with a genre with no labels. These will be cleaned out.")
noGenreLabel.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledGenresOrNoGenres.csv")

artist,artistLabel,genre,genreLabel
str,str,str,str
"""Q21546147""","""Patrick Chartol""",null,null
"""Q1468956""","""Marc Zermati""",null,null
"""Q106413686""","""Bryonn Bain""",null,null
"""Q11455087""","""Shōji Miyazawa""",null,null
"""Q54932941""","""Yumiko Yokoi""",null,null
…,…,…,…
"""Q95901533""","""Fritz Rating""",null,null
"""Q7984968""","""West Digges""",null,null
"""Q128694041""","""YNY Sebi""",null,null


In [60]:
#Record Labels without a label
noRecordLabelLabel = artistDF.filter(
    pl.col("recordLabel").is_in(pl.col("recordLabelLabel").implode())
).select(pl.col(["recordLabelLabel"])).unique()
recordLabel_dict = {k: None for k in noRecordLabelLabel["recordLabelLabel"].to_list()}
artistDF = artistDF.with_columns(
    pl.col("recordLabelLabel").replace(recordLabel_dict),
    recordLabel = pl.when(~pl.col('recordLabel').str.starts_with("Q")).then(None).otherwise(pl.col('recordLabel')),
)

recordLabelsWithNoLabel = artistDF.filter(
    pl.col("recordLabelLabel").is_null()
).select(["artist","artistLabel","recordLabel","recordLabelLabel"]).unique()
display(recordLabelsWithNoLabel)

recordLabelsWithNoLabel.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledRecordLabelsOrNoRecordLabels.csv")

artist,artistLabel,recordLabel,recordLabelLabel
str,str,str,str
"""Q94835215""","""Axel von Fielitz-Coniar""",null,null
"""Q112161628""","""Virgilio Blasi""",null,null
"""Q16407840""","""Igors Graps""",null,null
"""Q25189275""","""Vivek""",null,null
"""Q3040446""","""Esther Oosterbeek""",null,null
…,…,…,…
"""Q7494061""","""Shem von Schroeck""",null,null
"""Q2166294""","""Roque Ceruti""",null,null
"""Q41795525""","""Abdolreza Helali""",null,null


In [61]:
#Influences Labels without a label
noLabelsInfluencedBY = artistDF.filter(
    pl.col("influencedBY").is_in(pl.col("influencedBYLabel").implode())
).select(pl.col(["influencedBYLabel"])).unique()

influencedBName_dict = {k: None for k in noLabelsInfluencedBY["influencedBYLabel"].to_list()}
artistDF = artistDF.with_columns(
    pl.col("influencedBYLabel").replace(influencedBName_dict),
    influencedBY = pl.when(~pl.col('influencedBY').str.starts_with("Q")).then(None).otherwise(pl.col('influencedBY'))
)

entitiesWithNullInfluences = artistDF.filter(
    pl.col("influencedBYLabel").is_null()
).select(pl.col(["artist","artistLabel","influencedBY","influencedBYLabel"])).unique()
display(entitiesWithNullInfluences)
entitiesWithNullInfluences.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledInfluencesOrNoInfluences.csv")

artist,artistLabel,influencedBY,influencedBYLabel
str,str,str,str
"""Q115860138""","""Àngel Pongiluppi i Fenollosa""",null,null
"""Q5180678""","""Craig Abaya""",null,null
"""Q126392911""","""Brigitta Steinbrecher""",null,null
"""Q7787318""","""Thomas Bak""",null,null
"""Q71423""","""Claus Clausen""",null,null
…,…,…,…
"""Q6019216""","""Nisse Nyberg""",null,null
"""Q349896""","""Fayruz""",null,null
"""Q131340956""","""Luna Morgenstern""",null,null


In [64]:
#Look at selection of the musicBrainz table for any anomalies
mBtable = artistDF.select(pl.col(["artist","artistLabel","musicBrainzID"])).unique()
mB_issues = mBtable.filter(
    pl.col("musicBrainzID").str.contains("http")
).unique()
display(mB_issues)


mB_issues.write_csv("./missingValuesStore/artistsEntitiesWithFormattingIssuesInTheirMusicBrainzID.csv")
mB_users = mBtable.filter(
    pl.col("musicBrainzID").str.contains("user")
)
display(mB_users)


#Because it is two people who had these errored user links in there I will drop those two rows as their info will continue to be in the dataset
artistDF = artistDF.filter(
    ~pl.col("musicBrainzID").str.contains("user"),
)

#Get the artist ID of the people with the links for theirs instead of the number
artistDF = artistDF.with_columns(
    musicBrainzID = pl.when(pl.col("musicBrainzID").str.contains("artist")).then(
        pl.col("musicBrainzID").str.split("/").list.last()
    ).otherwise(pl.col("musicBrainzID")),
)

#See a selection of artists to make sure changes were done correctly.
display(artistDF.filter((pl.col("artist") == "Q138387061") | (pl.col("artist") == "Q138413725") | (pl.col("artist") == "Q139919749")  ).unique())


artist,artistLabel,musicBrainzID
str,str,str
"""Q139919749""","""Biko Manus""","""https://musicbrainz.org/artist…"
"""Q139623092""","""Mert Grey""","""https://musicbrainz.org/artist…"
"""Q140085977""","""Confidence Survival Eyong""","""https://musicbrainz.org/artist…"
"""Q12216002""","""Saadoun Jaber""","""https://musicbrainz.org/artist…"
"""Q138589085""","""Cyprian Błażejczyk""","""https://musicbrainz.org/artist…"
"""Q140159484""","""Boyonset""","""https://musicbrainz.org/artist…"


artist,artistLabel,musicBrainzID
str,str,str


artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str
"""Q138413725""","""Q639669""","""Q6581097""","""musician""","""male""","""Q219""",null,null,null,"""Q193207""",null,"""732910a2-5ee1-494b-9989-61afd6…",null,null,"""Bulgaria""","""ambient music""","""Konstantin Popov"""
"""Q138413725""","""Q183945""","""Q6581097""","""record producer""","""male""","""Q219""",null,null,null,"""Q9778""",null,"""732910a2-5ee1-494b-9989-61afd6…",null,null,"""Bulgaria""","""electronic music""","""Konstantin Popov"""
"""Q138413725""","""Q639669""","""Q6581097""","""musician""","""male""","""Q219""",null,null,null,"""Q9778""",null,"""732910a2-5ee1-494b-9989-61afd6…",null,null,"""Bulgaria""","""electronic music""","""Konstantin Popov"""
"""Q138387061""","""Q639669""","""Q6581097""","""musician""","""male""","""Q142""",null,null,null,"""Q223840""",null,"""f681a0d9-b9bb-4555-9b22-ada68f…",null,null,"""France""","""hardstyle""","""Monster Fragment"""
"""Q138413725""","""Q183945""","""Q6581097""","""record producer""","""male""","""Q219""",null,null,null,"""Q193207""",null,"""732910a2-5ee1-494b-9989-61afd6…",null,null,"""Bulgaria""","""ambient music""","""Konstantin Popov"""
"""Q139919749""","""Q183945""","""Q6581097""","""record producer""","""male""","""Q30""",null,null,null,null,null,"""0984057b-5ef5-4c92-80fa-585934…",null,null,"""United States""",null,"""Biko Manus"""


In [84]:
cleanedArtistDF = artistDF.filter(
    ~pl.col("artist").is_in(pl.col("artistLabel").implode()),
    ~pl.col("influencedBY").is_in(pl.col("influencedBYLabel").implode()),
    ~pl.col("recordLabel").is_in(pl.col("recordLabelLabel").implode()),
    ~pl.col("genre").is_in(pl.col("genreLabel").implode()),
)

#Add a column for presence of wikiPage
cleanedArtistDF = cleanedArtistDF.with_columns(
    isThereAWiki = pl.col("article").is_not_null()
)
#Finally de-duplicate the data.
cleanedArtistDF = cleanedArtistDF.unique()
display(cleanedArtistDF.head(5))
display(cleanedArtistDF.shape)

artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel,isThereAWiki
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,bool
"""Q194220""","""Q36834""","""Q6581097""","""composer""","""male""","""Q30""","""Q364214""","""Rakim""",null,"""Q753679""","""https://en.wikipedia.org/wiki/…","""cfbc0924-0035-4d6c-8197-f02465…","""Q183387""","""Columbia Records""","""United States""","""gangsta rap""","""Nas""",true
"""Q10681""","""Q36834""","""Q6581097""","""composer""","""male""","""Q142""","""Q3176402""","""Jef Le Penven""",222.0,"""Q910606""","""https://en.wikipedia.org/wiki/…","""abc61045-7ae3-48d9-91f2-d44689…","""Q21077""","""Warner Music Group""","""France""","""music of Brittany""","""Alan Stivell""",true
"""Q15123969""","""Q488205""","""Q48270""","""singer-songwriter""","""non-binary""","""Q145""","""Q182725""","""Etta James""",192.0,"""Q37073""","""https://en.wikipedia.org/wiki/…","""5a85c140-dcf9-4dd2-b2c8-aff047…","""Q193023""","""Capitol Records""","""United Kingdom""","""pop music""","""Sam Smith""",true
"""Q44479972""","""Q753110""","""Q6581097""","""songwriter""","""male""","""Q30""","""Q6107""","""Tupac Shakur""",null,"""Q15697534""","""https://en.wikipedia.org/wiki/…","""2282c539-05a2-477c-82b7-fe4079…","""Q59210322""","""Scumgang Records""","""United States""","""drill""","""6ix9ine""",true
"""Q230454""","""Q639669""","""Q6581072""","""musician""","""female""","""Q16""","""Q272913""","""Kitty Wells""",256.0,"""Q37073""","""https://en.wikipedia.org/wiki/…","""bf4bb21e-759f-4b22-bfd4-174593…","""Q843402""","""Warner Bros. Records""","""Canada""","""pop music""","""k.d. lang""",true


(136735, 18)

That is the end of the cleaning stage. Now time to analyse the data. Null values were left intentionally as we are looking for the gaps

In [85]:
#Discover the gender gap in the set.
artistGender = cleanedArtistDF.select(
    pl.col("artist"),
    pl.col("gender"),
    pl.col("genderLabel"),
)
artistGender = artistGender.unique()
artistGenderGrouped = artistGender.group_by(["gender", "genderLabel"]).agg(pl.col("artist").count())
artistGenderGroupedFRAC = artistGender["genderLabel"].value_counts(sort=True,normalize=True,name="fraction")
display(artistGenderGrouped)
display(artistGenderGroupedFRAC)

gender,genderLabel,artist
str,str,u32
"""Q6581097""","""male""",303
"""Q18116794""","""genderfluid""",4
"""Q1052281""","""trans woman""",1
"""Q6581072""","""female""",188
"""Q48270""","""non-binary""",4


genderLabel,fraction
str,f64
"""male""",0.606
"""female""",0.376
"""non-binary""",0.008
"""genderfluid""",0.008
"""trans woman""",0.002


In [86]:
genderChart = (
    alt.Chart(artistGenderGrouped).mark_bar().encode(
    y="artist",
    x="genderLabel",
    )
    .properties(width=500, title="Gender Distribution of Wikidata")
)
genderChart.encoding.x.title = "Genders"
genderChart.encoding.y.title = "Counts"
genderChart.show()



genderBase = alt.Chart(artistGenderGrouped).encode(
    alt.Theta("artist:Q").stack(True),
    alt.Color("genderLabel:N").legend(None)
    )
pie = genderBase.mark_arc(outerRadius=120)
text = genderBase.mark_text(radius=140, size=20).encode(text="genderLabel:N")
pie+text

alt.Chart(...)

alt.LayerChart(...)

In [87]:
#Top regions and region gap

regions = cleanedArtistDF.select(
    pl.col("artist"),
    pl.col("countryCitizenship"),
    pl.col("countryCitizenshipLabel"),
)

regions = regions.unique()
regionsGrouped = regions.group_by(["countryCitizenship", "countryCitizenshipLabel"]).agg(pl.col("artist").count())
display(regionsGrouped.head(5))

countryCitizenship,countryCitizenshipLabel,artist
str,str,u32
"""Q41""","""Greece""",5
"""Q159""","""Russia""",3
"""Q948""","""Tunisia""",1
"""Q884""","""South Korea""",1
"""Q33""","""Finland""",1


In [88]:
#Top genres and genre gap

In [89]:
#Top occupations

In [90]:
#Top occupations by gender

In [91]:
#Gender and genre split